In [1]:
from preprocessing import *
import joblib
import os
from sklearn.svm import SVC
from sklearn.metrics import mean_absolute_error, r2_score
def run_pipeline(df):
    categorical_cols, numeric_cols, high_missing_col, mid_missing_col, low_missing_col, single_label_cols = classify_columns(df)
    
    X_train, X_val, X_test, y_train, y_val, y_test, mid_missing_col, numeric_cols, multiLable_cols, categorical_cols = split_df(
        df, mid_missing_col, categorical_cols, numeric_cols, high_missing_col, low_missing_col
    )
    # Fill Missing Values
    X_train, X_val, X_test, categorical_cols, numeric_cols, it_imput = Full_Cleaning_Pipeline(X_train, X_val, X_test, multiLable_cols,single_label_cols)
    
    # # Encoding
    X_train, X_val, X_test, age_ed_enc = apply_ordinal_encoding(X_train, X_val, X_test)
    X_train, X_val, X_test, ohe_enc = apply_onehot_encoding(X_train, X_val, X_test)
    X_train, X_val, X_test, target_enc = apply_bayesian_encoding(X_train, X_val, X_test, y_train)
    
    ##--------------------- To Be Revisied
    multi_hot_encoders = {} 
    for col in multiLable_cols:
        X_train, X_val, X_test, mlb = apply_multihot_encoding(X_train, X_val, X_test, col)
        multi_hot_encoders[col] = mlb

    current_numeric = X_train.select_dtypes(include=['number']).columns.tolist()
    X_train, X_val, X_test, scaler = scale(X_train, X_val, X_test, current_numeric)
    
    # Selection
    X_train, X_val, X_test, var_selector = selecting_variance(X_train, X_val, X_test)
    X_train, X_val, X_test = select_randomForst(X_train, X_val, X_test, y_train)

    preprocessing_assets = {
    'imputer': it_imput,
    'years_code_median': X_train['YearsCode'].median(), 
    'country_salary_map': X_train.groupby('Country')['salary_log'].median(),
    'general_salary_median': X_train['salary_log'].median(),
    'tool_medians': {col: X_train[col].median() for col in ['ToolCountWork', 'ToolCountPersonal']},'scaler': scaler,
    'ordinal_encoder': age_ed_enc,
    'onehot_encoder': ohe_enc,
    'multi_hot_encoders': multi_hot_encoders,
    'variance_selector': var_selector,
    'final_features_names': X_train.columns.tolist(),
    'high_missing_col': high_missing_col
    }
    

    joblib.dump(preprocessing_assets, 'preprocessing_assets.pkl')
    print("All preprocessing assets have been saved successfully!")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

file_path =  "..\\data\\train_data_before_up.csv"
df = pd.read_csv(file_path)
X_train, X_val, X_test, y_train, y_val, y_test = run_pipeline(df)

Number of features before Variance Threshold:  1036
Number of features after Variance Threshold:  1036
                                               Feature  Importance
114  NewRole_i have strongly considered changing my...    0.080313
113  NewRole_i have somewhat considered changing my...    0.044116
40                                          salary_log    0.018700
87                             Employment_not employed    0.017659
0                                           ResponseId    0.016073
28                                             Country    0.015198
3                                              WorkExp    0.014130
4                                            YearsCode    0.013318
107  PurchaseInfluence_yes, i influenced the purcha...    0.012584
5                                              DevType    0.010771
26                                       ToolCountWork    0.008789
27                                   ToolCountPersonal    0.007953
111                       